# WTI Producer Hedge Simulator

**Business question:** How much can a crude producer reduce revenue volatility by hedging expected production with WTI futures?

A producer is naturally long physical crude. This notebook tests how short WTI futures can reduce that revenue risk.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import yfinance as yf
from pandas_datareader import data as web

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

from src.hedge_engine import HedgeAssumptions, prepare_monthly_market_data, compare_hedge_ratios, stress_test
from src.basis_risk import compare_basis_scenarios, simulate_basis_scenario, BasisHedgeAssumptions
from src.min_variance import minimum_variance_hedge_ratio, hedge_ratio_diagnostics, rolling_minimum_variance_hedge_ratio, contracts_for_hedge_ratio


## Download and prepare WTI data

In [ ]:
spot = web.DataReader('DCOILWTICO', 'fred', '2018-01-01')['DCOILWTICO']
raw = yf.download('CL=F', start='2018-01-01', auto_adjust=False, progress=False)
futures = raw['Close'].iloc[:, 0] if isinstance(raw.columns, pd.MultiIndex) else raw['Close']
market = prepare_monthly_market_data(spot, futures)
market.tail()


## Compare hedge ratios for 100,000 barrels/month

In [ ]:
assumptions = HedgeAssumptions(monthly_production_bbl=100_000)
summary, simulations = compare_hedge_ratios(market, assumptions=assumptions)
summary


## Stress test: 25% crude-price decline with a 75% hedge

In [ ]:
stress_test(spot_price=75, futures_entry=76, spot_shock_pct=-0.25, hedge_ratio=0.75, assumptions=assumptions)


## Basis risk: Midland physical crude hedged with Cushing WTI futures

A producer can hedge outright WTI price risk and still retain location basis risk. Here, basis is defined as **Midland minus Cushing**. The scenarios below are illustrative rather than a historical Midland cash series.

In [ ]:
basis_scenarios = compare_basis_scenarios(
    realized_basis_values=(1, 0, -1, -3, -5, -10),
    basis_hedge_ratios=(0, 0.50, 1.00),
    cushing_futures_entry=75,
    cushing_futures_exit=60,
    cushing_spot_exit=60,
    locked_basis_per_bbl=-1,
    flat_price_hedge_ratio=1.0,
    assumptions=assumptions,
)
basis_scenarios[[
    'basis_hedge_ratio',
    'realized_midland_basis',
    'basis_swap_pnl',
    'residual_revenue_risk',
]]


### Interpretation

With a full flat-price hedge but no basis hedge, a wider Midland discount still reduces realized revenue. A basis swap is a separate hedge designed to offset that location differential.

## Minimum-variance hedge ratio

Estimate the hedge ratio that minimizes the variance of monthly spot-price changes after the futures hedge.

In [ ]:
mv_ratio = minimum_variance_hedge_ratio(market)
mv_contracts = contracts_for_hedge_ratio(100_000, mv_ratio)
mv_ratio, mv_contracts


In [ ]:
hedge_ratio_diagnostics(market, mv_contracts / 100)


In [ ]:
rolling_mv = rolling_minimum_variance_hedge_ratio(market, window=24)
rolling_mv.tail()
